# 🧍 Human Pose Estimation с YOLO26 — практическое занятие

**Цель:** научиться находить ключевые точки тела человека на изображении и видео и анализировать результаты модели.

**Что вы сделаете:** запустите готовую модель → посмотрите 17 точек COCO → сравните входное изображение и результат → загрузите собственное изображение/видео → сохраните и скачаете обработанное видео.

**Нужно:** Google Colab (желательно GPU, но короткий пример работает и на CPU). **Обучение модели не требуется.**

⏱ Ориентировочно 45–60 минут. Запускайте ячейки **сверху вниз** (`Shift+Enter`).

## 0. Что такое pose estimation?

**Object Detection** находит человека и выдаёт *bounding box*. **Human Pose Estimation (HPE)** дополнительно определяет положения важных точек тела (плечи, локти, колени и т. д.). Линии между точками образуют **скелет**.

- Координаты `x, y` задаются в пикселях; начало координат — левый верхний угол.
- Каждая поза относится к **одному обнаруженному человеку**.
- Это **2D pose estimation**: по одной картинке нельзя достоверно восстановить 3D-положение человека.
- Когда тело перекрыто или человек далеко, некоторые точки могут определяться неточно.

**Вопрос перед запуском:** чем результат pose estimation отличается от instance segmentation?

## 1. Подготовка Colab

В меню **Runtime → Change runtime type** при необходимости выберите T4 GPU. Установите библиотеку (если после установки появляется просьба перезапустить runtime, выполните её и повторите запуск сверху).

In [ ]:
%pip -q install -U ultralytics opencv-python-headless matplotlib pillow

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from IPython.display import display, Video, HTML
from ultralytics import YOLO

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

WORK_DIR = Path('/content/pose_lesson')
WORK_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'yolo26n-pose.pt'  # n = лёгкая версия
CONF = 0.25
model = YOLO(MODEL_NAME)
print('Model loaded:', MODEL_NAME)

## 2. Первый пример: предсказание по готовой фотографии

На изображении с сайта Ultralytics модель должна найти людей и их ключевые точки. `result.plot()` рисует найденные точки, линии скелета и bounding boxes. Для отображения важно перевести BGR → RGB.

In [ ]:
example_url = 'https://ultralytics.com/images/bus.jpg'
result = model.predict(source=example_url, conf=CONF, device=DEVICE, verbose=False)[0]

original_rgb = cv2.cvtColor(result.orig_img, cv2.COLOR_BGR2RGB)
annotated_rgb = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(original_rgb)
axes[0].set_title('Source image')
axes[1].imshow(annotated_rgb)
axes[1].set_title('YOLO26 Pose prediction')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

n_people = 0 if result.keypoints is None else len(result.keypoints.xy)
print(f'Detected people: {n_people}')

## 3. Как читать 17 точек COCO

Для каждой обнаруженной персоны модель возвращает до 17 ключевых точек (формат COCO). **Левая/правая** сторона указана относительно самого человека, а не зрителя.

| Индексы | Части тела |
|---|---|
| 0–4 | нос, глаза, уши |
| 5–10 | плечи, локти, запястья |
| 11–16 | бёдра, колени, лодыжки |

`keypoints.xy` → координаты `(N,17,2)`, `keypoints.xyn` → нормализованные координаты от 0 до 1, `keypoints.data` → `(N,17,2)` или `(N,17,3)` в зависимости от модели (третий канал обычно confidence).

In [ ]:
KEYPOINT_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle'
]

if result.keypoints is None or len(result.keypoints.xy) == 0:
    print('No people detected. Try lowering CONF or choosing another image.')
else:
    xy = result.keypoints.xy.cpu().numpy()
    xyn = result.keypoints.xyn.cpu().numpy()
    kpts = result.keypoints.data.cpu().numpy()
    print('Pixel coordinates:', xy.shape)
    print('Normalised coordinates:', xyn.shape)
    print('Full keypoint data:', kpts.shape)
    print('\nFirst person (first 17 keypoints):')
    for idx, name in enumerate(KEYPOINT_NAMES):
        x, y = xy[0, idx]
        conf_text = f', score={kpts[0,idx,2]:.2f}' if kpts.shape[-1] > 2 else ''
        print(f'{idx:2d} {name:17s} x={x:6.1f}, y={y:6.1f}{conf_text}')

## 4. Изучаем точки визуально

Ниже каждая точка первого человека подписана номером. Сопоставьте номера с таблицей выше.

**Обратите внимание:** `(0,0)` иногда используется для точек, которые модель считает невидимыми/ненадёжными; не следует воспринимать их как реальное положение сустава.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(original_rgb)
if result.keypoints is not None and len(result.keypoints.xy) > 0:
    person_points = result.keypoints.xy[0].cpu().numpy()
    for i, (x, y) in enumerate(person_points):
        if x <= 0 and y <= 0:
            continue
        ax.scatter(x, y, s=35, c='yellow', edgecolors='black')
        ax.text(x + 5, y - 5, str(i), color='white', fontsize=10,
                bbox=dict(facecolor='black', alpha=0.7, pad=1))
ax.set_title('17 COCO keypoints (first detected person)')
ax.axis('off')
plt.show()

## 5. Своя фотография (необязательно)

Загрузите JPG/PNG с людьми или пропустите этот раздел. Вы увидите **Source | Prediction** и количество найденных людей. Colab может попросить разрешить загрузку файлов.

In [ ]:
from google.colab import files

print('Choose one image (JPG/PNG), or cancel upload to skip.')
uploaded_image = files.upload()
if uploaded_image:
    image_name = next(iter(uploaded_image))
    image_path = WORK_DIR / Path(image_name).name
    image_path.write_bytes(uploaded_image[image_name])
    print('Saved image:', image_path)
else:
    image_path = None
    print('Skipped custom image.')

In [ ]:
if image_path is not None:
    custom_result = model.predict(source=str(image_path), conf=CONF, device=DEVICE, verbose=False)[0]
    source = cv2.cvtColor(custom_result.orig_img, cv2.COLOR_BGR2RGB)
    prediction = cv2.cvtColor(custom_result.plot(), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(source); axes[0].set_title('Source')
    axes[1].imshow(prediction); axes[1].set_title('Prediction')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
    print('People:', 0 if custom_result.keypoints is None else len(custom_result.keypoints.xy))
else:
    print('Upload an image in the previous cell to run this example.')

## 6. Обработка своего видео

Загрузите короткий ролик (рекомендуется **MP4, 10–30 секунд**, с видимыми людьми). Если видео длинное, будет обработано только первые `MAX_FRAMES` кадров.

Здесь мы **самостоятельно обрабатываем кадры через OpenCV**: читаем кадр → запускаем YOLO Pose → рисуем предсказание → записываем MP4. Это позволяет не зависеть от внутренних имён папок `runs/...` и от исходного названия файла.

⚠️ Видео хранится в среде Colab и обычно удаляется после завершения сессии. Не загружайте конфиденциальные записи без необходимых разрешений.

In [ ]:
print('Choose one video (MP4, AVI, MOV), or cancel to skip.')
uploaded_video = files.upload()
if uploaded_video:
    video_name = next(iter(uploaded_video))
    video_path = WORK_DIR / Path(video_name).name
    video_path.write_bytes(uploaded_video[video_name])
    print('Video saved:', video_path)
else:
    video_path = None
    print('Skipped custom video.')

### 6.1 Параметры обработки

- `CONF` — минимальный порог уверенности обнаружения человека;
- `MAX_FRAMES` — ограничение объёма вычислений для занятия; `None` — обработать всё видео;
- `SHOW_BOXES` — показывать ли bounding boxes вместе со скелетом.

**Совет:** начните со 150 кадров и лёгкой модели `yolo26n-pose.pt`.

In [ ]:
MAX_FRAMES = 150   # Set None to process the full video
SHOW_BOXES = True
OUTPUT_VIDEO = WORK_DIR / 'pose_result_raw.mp4'

if video_path is None:
    print('Upload a video in section 6 to process it.')
else:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')

    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if np.isfinite(fps) and fps > 0 else 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width <= 0 or height <= 0:
        cap.release()
        raise RuntimeError('Invalid video dimensions')

    writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('Unable to create MP4 output with OpenCV')

    frames_processed = 0
    people_per_frame = []
    try:
        while True:
            if MAX_FRAMES is not None and frames_processed >= MAX_FRAMES:
                break
            ok, frame = cap.read()
            if not ok:
                break
            res = model.predict(source=frame, conf=CONF, device=DEVICE, verbose=False)[0]
            annotated = res.plot(boxes=SHOW_BOXES)
            writer.write(annotated)
            people_per_frame.append(0 if res.boxes is None else len(res.boxes))
            frames_processed += 1
            if frames_processed % 50 == 0:
                print(f'Processed {frames_processed} frames')
    finally:
        cap.release()
        writer.release()

    print(f'Processed frames: {frames_processed}')
    print(f'Output: {OUTPUT_VIDEO}')
    if frames_processed == 0:
        raise RuntimeError('No frames processed: check the input video codec')

### 6.2 Воспроизведение в Colab и скачивание

Некоторые браузеры не воспроизводят `mp4v`-файлы, поэтому для предпросмотра конвертируем результат в **H.264** с помощью FFmpeg (установлен в Colab). Выходное имя формируется автоматически.

In [ ]:
import subprocess

if video_path is not None and OUTPUT_VIDEO.exists():
    output_h264 = WORK_DIR / 'pose_result.mp4'
    proc = subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(OUTPUT_VIDEO),
         '-c:v', 'libx264', '-preset', 'veryfast', '-pix_fmt', 'yuv420p',
         '-movflags', '+faststart', str(output_h264)],
        capture_output=True, text=True
    )
    if proc.returncode != 0:
        print('H.264 conversion failed:', proc.stderr[-1200:])
        output_h264 = OUTPUT_VIDEO
    print('Ready:', output_h264)
    display(Video(str(output_h264), embed=True, width=720))
else:
    print('No processed video yet. Run section 6.1 first.')

In [ ]:
if video_path is not None and 'output_h264' in globals() and output_h264.exists():
    files.download(str(output_h264))
else:
    print('Process a video first to enable downloading.')

## 7. Мини-анализ результата

Задача: посчитать **число обнаруженных людей в каждом кадре**. Это ещё не трекинг: здесь у нас нет устойчивого идентификатора человека между кадрами.

График ниже строится только после обработки видео.

In [ ]:
if 'people_per_frame' in globals() and people_per_frame:
    plt.figure(figsize=(12, 3))
    plt.plot(range(len(people_per_frame)), people_per_frame)
    plt.xlabel('Frame index')
    plt.ylabel('Detected people')
    plt.title('Detections per frame (not unique people)')
    plt.grid(alpha=0.3)
    plt.show()
    print('Average people/frame:', round(float(np.mean(people_per_frame)), 2))
    print('Maximum people/frame:', max(people_per_frame))
else:
    print('Process a video to see the plot.')

## 8. Практические задания 📝

**Базовый уровень**

1. Измените `CONF` с `0.25` на `0.5`. Стало ли меньше ложных детекций? Не пропали ли люди?
2. Загрузите фотографию, где человек виден частично. Какие ключевые точки определяются хуже?
3. Найдите в выводе координаты **левого и правого запястий** (`9` и `10`). Какие значения получаются?

**Дополнительный уровень**

4. Замените `MODEL_NAME` на `yolo26s-pose.pt` и сравните результаты на одном изображении. Какая модель работает медленнее на вашем устройстве?
5. Нарисуйте на изображении только точки плеч (`5`, `6`) и колен (`13`, `14`).
6. Объясните, почему количество людей на каждом кадре нельзя сложить, чтобы узнать число уникальных людей на видео.

### Вопросы для самопроверки

- Чем pose estimation отличается от detection и segmentation?
- Что означают `(N, 17, 2)` и `(N, 17, 3)`?
- Почему модель может ошибаться при перекрытиях?
- Зачем нужны координаты `xyn`, если есть `xy`?

**Ожидаемый результат:** собственное изображение с ключевыми точками + скачанный MP4 с результатами + короткие выводы по экспериментам.

## 9. Материалы и связь со следующими темами

- [Ultralytics — Pose Estimation](https://docs.ultralytics.com/tasks/pose/)
- [Ultralytics — Predict](https://docs.ultralytics.com/modes/predict/)
- [COCO Keypoints](https://cocodataset.org/#keypoints-2020)

**Дальше:** Object Tracking добавляет устойчивые Track ID между кадрами. Pose estimation сама по себе определяет суставы на отдельных изображениях, но не гарантирует сохранение личности между кадрами.